# 04 · J.A.R.V.I.S. – your first agent

Dummy answered. **J.A.R.V.I.S.** *acts*: it has a name, standing instructions, tools and a memory, and it lives in Foundry, not in your script.

A **prompt agent** is configuration that Foundry runs for you:

| 🧩 Part | 🔎 Meaning |
|---|---|
| **name** | the stable identity, e.g. `jarvis` |
| **version** | every `create_version` call makes a new immutable version |
| **definition** | model + instructions + tools |
| **conversation** | the memory you learned in chapter 02 |

Deep dive: [docs/agents.md](../docs/agents.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

from azure.ai.projects.models import PromptAgentDefinition

AGENT = "jarvis-nb04"
oai = openai_client()

## 1 · Create the agent (version 1)

In [ ]:
agent = project().agents.create_version(
    agent_name=AGENT,
    definition=PromptAgentDefinition(
        model=model("chat"),
        instructions="You are J.A.R.V.I.S., Tony Stark's assistant. Be concise and dry. Address the user as 'sir'.",
    ),
    description="Chapter 04 demo agent",
)
print(f"id={agent.id}  name={agent.name}  version={agent.version}")

## 2 · Talk to it
Create a conversation, then point each response at the agent with `agent_reference`. The conversation keeps the history; the agent supplies model, instructions and tools.

In [ ]:
conv = oai.conversations.create()
for text in ["Good morning, J.A.R.V.I.S. Who are you?", "What did I just ask you?"]:
    r = oai.responses.create(conversation=conv.id, input=text, extra_body=agent_ref(agent))
    print(f"> {text}\n{r.output_text}\n")

### The shortcut: a client bound to the agent
`get_openai_client(agent_name=...)` sends the `agent_reference` for you.

In [ ]:
bound = openai_client(agent_name=AGENT)
print(bound.responses.create(conversation=conv.id, input="Summarize our conversation in six words.").output_text)

## 3 · Versions
Same name, new definition = new version. Old versions stay; calls without a version use the latest. This is how you roll out (and roll back) behaviour without touching client code.

In [ ]:
v2 = project().agents.create_version(
    agent_name=AGENT,
    definition=PromptAgentDefinition(
        model=model("chat"),
        instructions="You are J.A.R.V.I.S. Answer ONLY in rhyming couplets. Address the user as 'sir'.",
    ),
)
print("new version:", v2.version)
r = oai.responses.create(input="How is the weather in Malibu?", extra_body=agent_ref(AGENT))
print(r.output_text)

for v in project().agents.list_versions(agent_name=AGENT):
    print("version", v.version)

## 4 · Structured inputs (templated instructions)
Instructions can contain `{{placeholders}}` that you fill **per request** – one agent, many personalised behaviours.

In [ ]:
from azure.ai.projects.models import StructuredInputDefinition

tpl = project().agents.create_version(
    agent_name=AGENT,
    definition=PromptAgentDefinition(
        model=model("chat"),
        instructions="You are J.A.R.V.I.S. Greet {{person}} and mention their title: {{title}}. One sentence.",
        structured_inputs={
            "person": StructuredInputDefinition(description="Who is speaking", required=True, schema={"type": "string"}),
            "title": StructuredInputDefinition(description="Their title", required=True, schema={"type": "string"}),
        },
    ),
)
r = oai.responses.create(
    input="Hello!",
    extra_body={**agent_ref(AGENT), "structured_inputs": {"person": "Pepper", "title": "CEO of Stark Industries"}},
)
print(r.output_text)

## 5 · Streaming

In [ ]:
for event in oai.responses.create(input="Give me a three-point launch checklist.", stream=True, extra_body=agent_ref(AGENT)):
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()

## 6 · See it in the portal
**Build → Agents → jarvis-nb04**: versions, the playground, **Traces** and **Monitor** (after you connect Application Insights – Bicep did). Conversations created here show up with their id.

## Clean up

In [ ]:
oai.conversations.delete(conv.id)
delete_agent(AGENT)

## Recap
| Concept | Call |
|---|---|
| Create / update | `agents.create_version(agent_name, definition=PromptAgentDefinition(...))` |
| Talk | `responses.create(conversation=..., extra_body={"agent_reference": ...})` |
| Personalise | `structured_inputs` + `{{placeholders}}` |
| Remove | `agents.delete_version(...)` |

> 💡 Next: [05 · Tools](05_tools_function_and_builtin.ipynb) – give J.A.R.V.I.S. hands.